# Conecta Beauty — CRUD no Azure Cosmos DB com ingestão do Data Lake

**Disciplina:** Programação em Banco de Dados II — Prof. Samuel Henrique
**Entrega 1:** Implementação de operações CRUD em banco NoSQL com preparação de dados para Data Lake

## Fluxo de dados deste notebook

```
Azure Data Lake (ADLS Gen2)          Python (Colab)                     Azure Cosmos DB (API MongoDB)
bronze/gerado_por_script/*.jsonl ─►  leitura ─► limpeza ─► padronização ─►  pedidos_canais_externos
(pedidos brutos de 4 canais)                                              logs_tecnicos (populada pelo backend)
```

Os pedidos vêm de **4 canais externos simulados** (Instagram, Facebook, Marketplace e Mercado Livre),
e **cada canal manda o documento num formato diferente** — nomes de campo, aninhamento e presença de
CPF/CNPJ mudam de um canal para outro. É justamente esse cenário que justifica o uso de um banco NoSQL:
cada documento é gravado **preservando a estrutura original do seu canal**, e ganha um bloco
`padronizado` com os campos comuns, que permite consultar todos os canais de uma vez.

## Onde cada critério da entrega está atendido

| Critério do professor | Seção |
|---|---|
| 1. Configuração do ambiente no Colab | 1 e 2 |
| 2. Conexão com o banco NoSQL | 3 |
| 3. Criação de coleções distintas | 4 |
| 4. CREATE a partir de arquivo / Data Lake | 5, 6 e 7 |
| 5. READ — consulta simples, filtros, operadores e buscas personalizadas | 8 |
| 6. Evidências de execução | saída de todas as células |
| 7. Código comentado e organizado | todo o notebook |
| 8. Notebook executável e reprodutível | segredos no painel do Colab + CREATE idempotente (pode rodar várias vezes) |

## 1. Ambiente

Instala as bibliotecas que não vêm no Colab por padrão:
- **pymongo** — cliente MongoDB, usado para conversar com o Azure Cosmos DB (API para MongoDB);
- **azure-storage-file-datalake** — SDK do Azure Data Lake Storage Gen2, usado para ler a camada bronze.

In [ ]:
!pip install -q pymongo azure-storage-file-datalake

import hashlib
import json
import re
import time
import unicodedata
import warnings
from collections import Counter
from datetime import datetime, timedelta, timezone
from pprint import pprint

from azure.storage.filedatalake import FileSystemClient
from pymongo import MongoClient, ReplaceOne
from pymongo.errors import BulkWriteError, OperationFailure

# O pymongo avisa que está falando com um Cosmos DB (e não um MongoDB "puro").
# O aviso é esperado e só polui a saída, então é silenciado.
warnings.filterwarnings("ignore", message="You appear to be connected to a CosmosDB cluster")

print("Bibliotecas carregadas.")

## 2. Credenciais e configuração

**Nenhuma credencial fica escrita no código.** Elas são lidas do painel de *Secrets* do Colab
(ícone de chave 🔑 na barra lateral esquerda). Cadastre os dois segredos abaixo e ative o
acesso deste notebook a eles:

| Nome do segredo | Onde conseguir |
|---|---|
| `COSMOS_MONGO_CONNECTION_STRING` | Portal Azure → Cosmos DB → *Cadeia de conexão* |
| `DATALAKE_SAS_TOKEN` | Portal Azure → Storage account → Contêineres → `bronze` → *Gerar SAS* (permissões **Leitura** e **Lista**, com data de expiração) |

O token SAS dá acesso **somente de leitura** ao container `bronze` e expira sozinho — mais seguro do
que usar a chave da conta de armazenamento, que dá acesso total a tudo.

Se o segredo não estiver cadastrado, o notebook pede o valor pelo teclado, sem exibi-lo na tela.

In [ ]:
def ler_segredo(nome):
    # Tenta o painel de Secrets do Colab; se não houver, pede o valor sem ecoar na tela.
    try:
        from google.colab import userdata
        valor = userdata.get(nome)
        if valor:
            return valor
    except Exception:
        pass
    from getpass import getpass
    return getpass(f"Informe {nome}: ")


COSMOS_CONNECTION_STRING = ler_segredo("COSMOS_MONGO_CONNECTION_STRING")
DATALAKE_SAS_TOKEN = ler_segredo("DATALAKE_SAS_TOKEN")

print("Credenciais carregadas (os valores não são exibidos).")

In [ ]:
# Mesmo banco usado pela API do projeto: o backend (utils/mongo_logger.py) grava os
# logs técnicos em conectabeauty.logs_tecnicos. Usar outro nome criaria um banco separado.
NOME_BANCO = "conectabeauty"

NOME_COLECAO_PEDIDOS = "pedidos_canais_externos"  # criada e populada por este notebook
NOME_COLECAO_LOGS = "logs_tecnicos"               # populada automaticamente pela API — aqui só é lida

# Localização dos arquivos brutos no Data Lake.
DATALAKE_ACCOUNT_URL = "https://datalakeconectabeauty.dfs.core.windows.net"
DATALAKE_CONTAINER = "bronze"
DATALAKE_PASTA = "gerado_por_script"

# None = carrega todos os pedidos. Um número pequeno (ex.: 200) serve para um teste rápido.
LIMITE_DOCUMENTOS = None

# Quantidade de documentos gravados por requisição ao Cosmos DB.
TAMANHO_LOTE = 100

# True apaga e recria a coleção de pedidos antes da carga (começa do zero).
RECRIAR_COLECAO = False

## 3. Conexão com o Azure Cosmos DB

O comando `ping` confirma que a conexão realmente funciona antes de seguir.

In [ ]:
client = MongoClient(COSMOS_CONNECTION_STRING, serverSelectionTimeoutMS=10000)

try:
    client.admin.command("ping")
except Exception as erro:
    raise ConnectionError(f"Não foi possível conectar ao Azure Cosmos DB: {erro}") from erro

db = client[NOME_BANCO]

print("Conexão com o Azure Cosmos DB realizada com sucesso!")
print(f"Banco: {NOME_BANCO}")
print(f"Coleções existentes: {db.list_collection_names()}")

## 4. Criação das coleções

- **`pedidos_canais_externos`** é criada aqui, explicitamente.
- **`logs_tecnicos`** não é criada pelo notebook: ela nasce e é alimentada sozinha pela API do projeto
  (cada requisição e cada erro geram um documento). O notebook só consulta essa coleção na seção 8.

No Cosmos DB (API MongoDB 3.6+) apenas o `_id` é indexado por padrão, e ordenar por um campo sem índice
gera erro. Por isso são criados índices nos campos usados nas consultas.

In [ ]:
if RECRIAR_COLECAO and NOME_COLECAO_PEDIDOS in db.list_collection_names():
    db.drop_collection(NOME_COLECAO_PEDIDOS)
    print(f"Coleção '{NOME_COLECAO_PEDIDOS}' apagada para recriação.")

if NOME_COLECAO_PEDIDOS in db.list_collection_names():
    print(f"Coleção '{NOME_COLECAO_PEDIDOS}' já existe — mantida.")
else:
    db.create_collection(NOME_COLECAO_PEDIDOS)
    print(f"Coleção '{NOME_COLECAO_PEDIDOS}' criada.")

pedidos_col = db[NOME_COLECAO_PEDIDOS]
logs_col = db[NOME_COLECAO_LOGS]

# A carga grava cada pedido pelo _id. Se a coleção tiver sido criada antes com outra
# chave de partição (shard key), essa gravação não funciona — melhor avisar já aqui.
try:
    info = db.command({"customAction": "GetCollection", "collection": NOME_COLECAO_PEDIDOS})
    chave = info.get("shardKeyDefinition")
    print(f"Chave de partição: {chave or 'nenhuma (coleção não particionada)'}")
    if chave and list(chave) != ["_id"]:
        raise RuntimeError(
            f"A coleção foi criada com a chave de partição {chave}. "
            "Defina RECRIAR_COLECAO = True na seção 2 e rode novamente."
        )
except OperationFailure:
    print("Metadados de partição indisponíveis (comando exclusivo do Cosmos DB) — seguindo.")

INDICES = [
    "padronizado.canal",
    "padronizado.valor_total",
    "padronizado.data_pedido",
    "padronizado.forma_pagamento",
    "padronizado.estado",
]
for campo in INDICES:
    pedidos_col.create_index(campo)

print(f"Índices da coleção: {sorted(pedidos_col.index_information())}")

## 5. Leitura da camada bronze do Data Lake

O notebook percorre a pasta `bronze/gerado_por_script` (inclusive subpastas), encontra todos os arquivos
`.jsonl` e lê cada linha como um pedido.

> **Pré-requisito de rede:** a storage account aceita conexões apenas dos IPs liberados no firewall.
> O Colab roda em servidores do Google com IP variável, então em *Rede* a storage account precisa
> estar em "Habilitar de todas as redes" durante a execução — o acesso continua protegido pelo token SAS.

In [ ]:
fs = FileSystemClient(
    account_url=DATALAKE_ACCOUNT_URL,
    file_system_name=DATALAKE_CONTAINER,
    credential=DATALAKE_SAS_TOKEN,
)

arquivos = [
    caminho
    for caminho in fs.get_paths(path=DATALAKE_PASTA, recursive=True)
    if not caminho.is_directory and caminho.name.endswith(".jsonl")
]

if not arquivos:
    raise FileNotFoundError(f"Nenhum arquivo .jsonl encontrado em {DATALAKE_CONTAINER}/{DATALAKE_PASTA}.")

print(f"Arquivos .jsonl encontrados em {DATALAKE_CONTAINER}/{DATALAKE_PASTA}: {len(arquivos)}")
for caminho in arquivos:
    print(f" - {caminho.name} ({caminho.content_length / 1024:,.1f} KB, modificado em {caminho.last_modified})")

In [ ]:
registros_brutos = []  # cada item guarda o documento original e de onde ele veio

for caminho in arquivos:
    conteudo = fs.get_file_client(caminho.name).download_file().readall().decode("utf-8")

    linhas_validas = linhas_invalidas = 0
    for numero_linha, linha in enumerate(conteudo.splitlines(), start=1):
        if not linha.strip():
            continue
        try:
            registros_brutos.append({"arquivo": caminho.name, "linha": numero_linha, "doc": json.loads(linha)})
            linhas_validas += 1
        except json.JSONDecodeError:
            linhas_invalidas += 1

    print(f"{caminho.name}: {linhas_validas} pedidos lidos, {linhas_invalidas} linhas inválidas descartadas")

if LIMITE_DOCUMENTOS:
    registros_brutos = registros_brutos[:LIMITE_DOCUMENTOS]
    print(f"\nLIMITE_DOCUMENTOS ativo: usando apenas os primeiros {len(registros_brutos)} pedidos.")

print(f"\nTotal de pedidos brutos carregados: {len(registros_brutos)}")

## 6. Tratamento dos dados

Cada canal identifica o pedido com um campo diferente, e é por ele que o canal é reconhecido.
O mapa abaixo diz, para cada canal, **onde** está cada informação no documento original.

In [ ]:
# Para cada canal: campo que identifica o canal + caminho de cada informação no documento.
# None = o canal não envia essa informação.
MAPA_CANAIS = {
    "Instagram": {
        "id": "pedido_instagram_id", "cliente": "cliente.nome", "telefone": "cliente.telefone",
        "cidade": "cliente.cidade", "estado": None, "documento": None, "itens": "itens",
        "valor": "valor_total", "frete": None, "pagamento": "forma_pagamento", "data": "data_pedido",
    },
    "Facebook": {
        "id": "id_pedido_fb", "cliente": "comprador_nome", "telefone": "comprador_telefone",
        "cidade": "cidade_entrega", "estado": None, "documento": "cpf_informado", "itens": "produtos",
        "valor": "total", "frete": None, "pagamento": "pagamento", "data": "criado_em",
    },
    "Marketplace": {
        "id": "order_id", "cliente": "comprador.nome", "telefone": "comprador.telefone",
        "cidade": "endereco_entrega.cidade", "estado": "endereco_entrega.estado",
        "documento": "comprador.documento", "itens": "itens_pedido", "valor": "valor_total_pedido",
        "frete": "valor_frete", "pagamento": "forma_pagamento", "data": "timestamp_pedido",
    },
    "Mercado Livre": {
        "id": "ml_order_id", "cliente": "buyer.full_name", "telefone": "buyer.phone",
        "cidade": "shipping.city", "estado": "shipping.state", "documento": "buyer.document",
        "itens": "items", "valor": "total_amount", "frete": "shipping.cost",
        "pagamento": "payment_method", "data": "order_date",
    },
}


def identificar_canal(doc):
    for canal, mapa in MAPA_CANAIS.items():
        if mapa["id"] in doc:
            return canal
    return None


def ler_caminho(doc, caminho):
    # Lê um campo aninhado. Ex.: ler_caminho(doc, "buyer.full_name")
    if caminho is None:
        return None
    atual = doc
    for parte in caminho.split("."):
        if not isinstance(atual, dict):
            return None
        atual = atual.get(parte)
    return atual


# Evidência de que os canais chegam em formatos diferentes.
exemplo_por_canal = {}
for registro in registros_brutos:
    canal = identificar_canal(registro["doc"])
    if canal and canal not in exemplo_por_canal:
        exemplo_por_canal[canal] = registro["doc"]

for canal, doc in exemplo_por_canal.items():
    print(f"\n=== {canal} — campos do documento original ===")
    print(list(doc.keys()))

### Limpeza

A limpeza é **genérica**: percorre o documento inteiro, em qualquer nível de aninhamento, sem depender
dos nomes de campo de cada canal. Ela:
- normaliza a codificação Unicode dos textos e remove espaços duplicados ou nas pontas;
- deixa só os dígitos em documentos (CPF/CNPJ) e telefones, removendo pontos, traços, parênteses e espaços.

Os acentos **não** são removidos: "São Paulo" e "Sao Paulo" são o mesmo texto para uma pessoa, mas
remover acento apaga informação do dado original e não resolve nenhum problema de consulta.

Documentos (CPF/CNPJ) são **validados pelos dígitos verificadores**, mas nunca inventados: se um canal não
informa documento (o Instagram, por exemplo), o campo continua ausente — essa ausência é uma
característica real daquele canal, não um erro a ser "corrigido".

In [ ]:
CAMPOS_SO_DIGITOS = {"documento", "document", "cpf_informado", "telefone", "phone", "comprador_telefone"}


def normalizar_texto(texto):
    texto = unicodedata.normalize("NFC", texto)
    return re.sub(r"\s+", " ", texto).strip()


def limpar_recursivo(valor, chave=None):
    # Percorre dicionários e listas em qualquer profundidade.
    if isinstance(valor, dict):
        return {k: limpar_recursivo(v, k) for k, v in valor.items()}
    if isinstance(valor, list):
        return [limpar_recursivo(v, chave) for v in valor]
    if isinstance(valor, str):
        if chave in CAMPOS_SO_DIGITOS:
            return re.sub(r"\D", "", valor)
        return normalizar_texto(valor)
    return valor


def cpf_valido(cpf):
    if len(cpf) != 11 or cpf == cpf[0] * 11:
        return False
    for posicao in (9, 10):
        soma = sum(int(cpf[i]) * (posicao + 1 - i) for i in range(posicao))
        if (soma * 10) % 11 % 10 != int(cpf[posicao]):
            return False
    return True


def cnpj_valido(cnpj):
    if len(cnpj) != 14 or cnpj == cnpj[0] * 14:
        return False
    pesos = [5, 4, 3, 2, 9, 8, 7, 6, 5, 4, 3, 2]
    for posicao, pesos_posicao in ((12, pesos), (13, [6] + pesos)):
        resto = sum(int(cnpj[i]) * pesos_posicao[i] for i in range(posicao)) % 11
        if (0 if resto < 2 else 11 - resto) != int(cnpj[posicao]):
            return False
    return True


def moeda(valor):
    # Formata no padrão brasileiro: 1234.5 -> "R$ 1.234,50"
    return "R$ " + f"{valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


print("Funções de limpeza, validação e formatação definidas.")

### Padronização

Cada documento ganha um bloco `padronizado` com as mesmas informações para todos os canais
(cliente, cidade, valor, forma de pagamento, data...). O documento original do canal é mantido
intacto ao lado — o bloco padronizado é um acréscimo, não uma substituição.

A data vira um tipo data de verdade (não texto), o que permite consultas por período.

Cada pedido recebe como `_id` um *hash* do seu conteúdo original. Com isso:
- pedidos repetidos no arquivo viram um só (**remoção de duplicidade**);
- rodar o notebook de novo **atualiza** os pedidos existentes em vez de duplicá-los.

In [ ]:
def padronizar(doc, canal):
    mapa = MAPA_CANAIS[canal]

    itens = ler_caminho(doc, mapa["itens"]) or []
    documento = ler_caminho(doc, mapa["documento"])
    valor = ler_caminho(doc, mapa["valor"])
    frete = ler_caminho(doc, mapa["frete"])

    try:
        data_pedido = datetime.fromisoformat(ler_caminho(doc, mapa["data"]))
    except (TypeError, ValueError):
        data_pedido = None

    tipo_documento = documento_ok = None
    if documento:
        tipo_documento = {11: "CPF", 14: "CNPJ"}.get(len(documento))
        documento_ok = cpf_valido(documento) if tipo_documento == "CPF" else cnpj_valido(documento)

    padronizado = {
        "canal": canal,
        "id_pedido_canal": str(ler_caminho(doc, mapa["id"])),
        "cliente": ler_caminho(doc, mapa["cliente"]),
        "telefone": ler_caminho(doc, mapa["telefone"]),
        "cidade": ler_caminho(doc, mapa["cidade"]),
        "estado": ler_caminho(doc, mapa["estado"]),
        "documento": documento,
        "tipo_documento": tipo_documento,
        "documento_valido": documento_ok,
        "produtos": [item.get("produto") for item in itens],
        "qtd_itens": sum(int(item.get("quantidade") or 0) for item in itens),
        "valor_total": round(float(valor), 2) if valor is not None else None,
        "frete": round(float(frete), 2) if frete is not None else None,
        "forma_pagamento": ler_caminho(doc, mapa["pagamento"]),
        "data_pedido": data_pedido,
    }

    # Problemas de qualidade são sinalizados, não "consertados" por conta própria.
    alertas = []
    if not padronizado["valor_total"]:
        alertas.append("valor_total ausente ou zerado")
    if not itens:
        alertas.append("pedido sem itens")
    if data_pedido is None:
        alertas.append("data do pedido ausente ou inválida")
    if documento and not documento_ok:
        alertas.append("documento inválido")
    padronizado["alertas_qualidade"] = alertas

    return padronizado


def tratar_registro(registro):
    bruto = registro["doc"]
    canal = identificar_canal(bruto)
    if canal is None:
        return None

    tratado = limpar_recursivo(bruto)
    tratado["_id"] = hashlib.sha1(json.dumps(bruto, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()
    tratado["padronizado"] = padronizar(tratado, canal)
    tratado["ingestao"] = {
        "origem": f"{DATALAKE_CONTAINER}/{registro['arquivo']}",
        "linha": registro["linha"],
        "carregado_em": datetime.now(timezone.utc),
        "alterado_na_limpeza": limpar_recursivo(bruto) != bruto,
    }
    return tratado

In [ ]:
pedidos_tratados = {}
rejeitados = 0

for registro in registros_brutos:
    tratado = tratar_registro(registro)
    if tratado is None:
        rejeitados += 1
        continue
    pedidos_tratados[tratado["_id"]] = tratado  # mesmo _id = mesmo pedido: fica um só

pedidos_tratados = list(pedidos_tratados.values())
duplicados = len(registros_brutos) - rejeitados - len(pedidos_tratados)

print("===== RESUMO DO TRATAMENTO =====")
print(f"Pedidos brutos lidos:          {len(registros_brutos)}")
print(f"Canal não identificado:        {rejeitados}")
print(f"Duplicados removidos:          {duplicados}")
print(f"Pedidos prontos para gravar:   {len(pedidos_tratados)}")
print(f"Alterados pela limpeza:        {sum(p['ingestao']['alterado_na_limpeza'] for p in pedidos_tratados)}")
print(f"Com alerta de qualidade:       {sum(bool(p['padronizado']['alertas_qualidade']) for p in pedidos_tratados)}")

print("\nPedidos por canal:")
for canal, quantidade in Counter(p["padronizado"]["canal"] for p in pedidos_tratados).most_common():
    print(f"  {canal:<15} {quantidade}")

In [ ]:
# Antes x depois de um pedido do Marketplace (o formato com mais campos para limpar).
exemplo = next(r for r in registros_brutos if identificar_canal(r["doc"]) == "Marketplace")
depois = tratar_registro(exemplo)

print("=== ANTES (como veio do Data Lake) ===")
pprint(exemplo["doc"]["comprador"])

print("\n=== DEPOIS (documento e telefone limpos) ===")
pprint(depois["comprador"])

print("\n=== BLOCO PADRONIZADO ADICIONADO ===")
pprint(depois["padronizado"])

## 7. CREATE — inserção no Cosmos DB

Os pedidos são gravados em lotes com `bulk_write` + `ReplaceOne(upsert=True)`:
- se o `_id` ainda não existe na coleção, o pedido é **inserido**;
- se já existe (o notebook já rodou antes), é **substituído** — nunca duplicado.

O Cosmos DB limita a vazão de escrita (RU/s). Quando esse limite é atingido, ele recusa temporariamente as
operações excedentes (erro 16500). A função abaixo espera um pouco e reenvia só o que foi recusado.

In [ ]:
COD_LIMITE_VAZAO = 16500  # "TooManyRequests" do Cosmos DB


def gravar_em_lotes(colecao, documentos, tamanho_lote=TAMANHO_LOTE, max_tentativas=10):
    novos = substituidos = 0

    for inicio in range(0, len(documentos), tamanho_lote):
        operacoes = [
            ReplaceOne({"_id": doc["_id"]}, doc, upsert=True)
            for doc in documentos[inicio:inicio + tamanho_lote]
        ]

        for tentativa in range(1, max_tentativas + 1):
            try:
                resultado = colecao.bulk_write(operacoes, ordered=False)
                novos += resultado.upserted_count
                substituidos += resultado.matched_count
                break
            except BulkWriteError as erro:
                detalhes = erro.details
                novos += detalhes.get("nUpserted", 0)
                substituidos += detalhes.get("nMatched", 0)

                erros = detalhes.get("writeErrors", [])
                if any(e.get("code") != COD_LIMITE_VAZAO for e in erros):
                    raise
                # Reenvia somente as operações recusadas por limite de vazão.
                operacoes = [operacoes[e["index"]] for e in erros]
            except OperationFailure as erro:
                if erro.code != COD_LIMITE_VAZAO:
                    raise
            time.sleep(min(0.5 * 2 ** tentativa, 30))
        else:
            raise RuntimeError(f"Lote iniciado em {inicio} excedeu {max_tentativas} tentativas.")

        gravados = min(inicio + tamanho_lote, len(documentos))
        if gravados % 1000 == 0 or gravados == len(documentos):
            print(f"  {gravados}/{len(documentos)} pedidos processados")

    return novos, substituidos


inicio_carga = time.time()
novos, substituidos = gravar_em_lotes(pedidos_col, pedidos_tratados)

print("\nCREATE executado com sucesso!")
print(f"Pedidos novos inseridos:           {novos}")
print(f"Pedidos já existentes, atualizados: {substituidos}")
print(f"Tempo total:                       {time.time() - inicio_carga:.1f}s")
print(f"Documentos na coleção agora:       {pedidos_col.count_documents({})}")

## 8. READ — consultas

As consultas vão do mais simples ao mais elaborado. Várias usam o bloco `padronizado`, que permite
comparar todos os canais juntos; outras consultam direto os campos originais de um canal específico,
mostrando que a estrutura de cada canal foi preservada.

In [ ]:
# 8.1 Consulta simples: total de documentos e um pedido completo.
print(f"Total de pedidos na coleção: {pedidos_col.count_documents({})}\n")

print("Um pedido qualquer, completo:")
pprint(pedidos_col.find_one())

In [ ]:
# 8.2 Filtro simples por igualdade: pedidos do Instagram.
filtro = {"padronizado.canal": "Instagram"}
print(f"Pedidos do Instagram: {pedidos_col.count_documents(filtro)}\n")

projecao = {"_id": 0, "padronizado.cliente": 1, "padronizado.cidade": 1, "padronizado.valor_total": 1}
for pedido in pedidos_col.find(filtro, projecao).limit(5):
    print(pedido["padronizado"])

In [ ]:
# 8.3 Consulta no formato original de um canal: pedidos do Mercado Livre
# buscados pelo campo aninhado "shipping.state", que só existe nesse canal.
filtro = {"channel": "Mercado Livre", "shipping.state": "SP"}
print(f"Pedidos do Mercado Livre com entrega em SP: {pedidos_col.count_documents(filtro)}\n")

for pedido in pedidos_col.find(filtro, {"_id": 0, "ml_order_id": 1, "buyer.full_name": 1, "shipping": 1}).limit(5):
    print(pedido)

In [ ]:
# 8.4 Operador $exists: no Facebook o cliente às vezes informa o CPF, às vezes não.
# Em um banco NoSQL o campo simplesmente não existe quando não foi informado.
total_facebook = pedidos_col.count_documents({"origem": "Facebook"})
com_cpf = pedidos_col.count_documents({"origem": "Facebook", "cpf_informado": {"$exists": True}})

print(f"Pedidos do Facebook:          {total_facebook}")
print(f"  com CPF informado:          {com_cpf}")
print(f"  sem o campo cpf_informado:  {total_facebook - com_cpf}")

In [ ]:
# 8.5 Operador $gt + ordenação + limite: os 5 maiores pedidos de todos os canais.
for pedido in (
    pedidos_col.find({"padronizado.valor_total": {"$gt": 10000}})
    .sort("padronizado.valor_total", -1)
    .limit(5)
):
    p = pedido["padronizado"]
    print(f"{moeda(p['valor_total']):>14} | {p['canal']:<13} | {p['cliente']}")

In [ ]:
# 8.6 Operador $in: pedidos pagos com Pix ou boleto.
filtro = {"padronizado.forma_pagamento": {"$in": ["Pix", "Boleto"]}}
print(f"Pedidos pagos com Pix ou boleto: {pedidos_col.count_documents(filtro)}")

In [ ]:
# 8.7 Intervalo de datas ($gte / $lt): pedidos feitos nos últimos 30 dias.
agora = datetime.now()
filtro = {"padronizado.data_pedido": {"$gte": agora - timedelta(days=30), "$lt": agora}}
print(f"Pedidos dos últimos 30 dias: {pedidos_col.count_documents(filtro)}")

In [ ]:
# 8.8 Operador $or com condições combinadas:
# Mercado Livre acima de R$ 5.000  OU  Marketplace com frete grátis.
filtro = {
    "$or": [
        {"padronizado.canal": "Mercado Livre", "padronizado.valor_total": {"$gt": 5000}},
        {"padronizado.canal": "Marketplace", "padronizado.frete": 0},
    ]
}
print(f"Pedidos que atendem a pelo menos uma das condições: {pedidos_col.count_documents(filtro)}")

In [ ]:
# 8.9 Expressão regular ($regex): clientes cujo nome começa com "Maria" (inclui Mariana,
# Mariane...), ignorando maiúsculas e minúsculas.
filtro = {"padronizado.cliente": {"$regex": "^maria", "$options": "i"}}
print(f"Pedidos de clientes cujo nome começa com 'Maria': {pedidos_col.count_documents(filtro)}\n")

for pedido in pedidos_col.find(filtro, {"_id": 0, "padronizado.cliente": 1, "padronizado.canal": 1}).limit(5):
    print(pedido["padronizado"])

In [ ]:
# 8.10 Consulta em lista ($size): pedidos com exatamente 4 produtos diferentes.
filtro = {"padronizado.produtos": {"$size": 4}}
print(f"Pedidos com 4 produtos diferentes: {pedidos_col.count_documents(filtro)}")

In [ ]:
# 8.11 Qualidade dos dados: pedidos que receberam algum alerta no tratamento.
filtro = {"padronizado.alertas_qualidade.0": {"$exists": True}}
print(f"Pedidos com alerta de qualidade: {pedidos_col.count_documents(filtro)}")

for pedido in pedidos_col.find(filtro, {"_id": 0, "padronizado.canal": 1, "padronizado.alertas_qualidade": 1}).limit(5):
    print(pedido["padronizado"])

In [ ]:
# 8.12 Agregação: faturamento, quantidade de pedidos e ticket médio por canal.
pipeline = [
    {"$group": {
        "_id": "$padronizado.canal",
        "pedidos": {"$sum": 1},
        "faturamento": {"$sum": "$padronizado.valor_total"},
        "ticket_medio": {"$avg": "$padronizado.valor_total"},
    }},
    {"$sort": {"faturamento": -1}},
]

print(f"{'Canal':<15}{'Pedidos':>9}{'Faturamento':>18}{'Ticket médio':>15}")
for linha in pedidos_col.aggregate(pipeline):
    print(f"{linha['_id']:<15}{linha['pedidos']:>9}{moeda(linha['faturamento']):>18}{moeda(linha['ticket_medio']):>15}")

In [ ]:
# 8.13 Agregação com filtro: os 5 estados com mais pedidos.
# Só Marketplace e Mercado Livre informam o estado — os demais canais ficam de fora do $match.
pipeline = [
    {"$match": {"padronizado.estado": {"$ne": None}}},
    {"$group": {"_id": "$padronizado.estado", "pedidos": {"$sum": 1}}},
    {"$sort": {"pedidos": -1}},
    {"$limit": 5},
]
for linha in pedidos_col.aggregate(pipeline):
    print(f"{linha['_id']}: {linha['pedidos']} pedidos")

### Coleção `logs_tecnicos`

Esta coleção é populada automaticamente pela API do Conecta Beauty: cada requisição recebida e cada erro
gera um documento. Aqui ela é apenas consultada.

In [ ]:
# 8.14 Leitura da segunda coleção: resumo por tipo de evento e os eventos mais recentes.
total_logs = logs_col.count_documents({})
print(f"Documentos em {NOME_COLECAO_LOGS}: {total_logs}")

if total_logs == 0:
    print("Nenhum log ainda — a coleção é preenchida quando a API do projeto recebe requisições.")
else:
    for linha in logs_col.aggregate([{"$group": {"_id": "$tipo_evento", "quantidade": {"$sum": 1}}}]):
        print(f"  {linha['_id']}: {linha['quantidade']}")

    print("\nÚltimos 5 eventos registrados:")
    for log in logs_col.find({}, {"_id": 0, "mensagem": 1}).sort("_id", -1).limit(5):
        print(f"  {log.get('mensagem')}")

## 9. Verificação final

In [ ]:
print("===== VERIFICAÇÃO FINAL =====")
print(f"Banco:                        {NOME_BANCO}")
print(f"Coleções:                     {sorted(db.list_collection_names())}")
print(f"Pedidos em {NOME_COLECAO_PEDIDOS}:  {pedidos_col.count_documents({})}")
print(f"Documentos em {NOME_COLECAO_LOGS}:  {logs_col.count_documents({})}")
print(f"Arquivos lidos do Data Lake:  {len(arquivos)}")
print()
print("CREATE -> pedidos lidos do Data Lake, tratados e gravados no Cosmos DB")
print("READ   -> filtros, projeção, $exists, $gt, $in, $or, $regex, $size, datas, ordenação e agregações")

client.close()